# Calculus & Optimization — AI Engineering Interview Prep

Covers: gradients, chain rule, gradient descent, momentum, Adam.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.style.use("seaborn-v0_8")
print("Ready")

## 1. Numerical Differentiation

In [ ]:
def numerical_gradient(f, x, h=1e-5):
    """Central difference: (f(x+h) - f(x-h)) / (2h)"""
    return (f(x + h) - f(x - h)) / (2 * h)

# Test on f(x) = x³ - 2x  →  f'(x) = 3x² - 2
f = lambda x: x**3 - 2*x
df_analytical = lambda x: 3*x**2 - 2

for x in [0, 1, 2, -1.5]:
    num = numerical_gradient(f, x)
    ana = df_analytical(x)
    print(f"x={x:5.1f}: numerical={num:.6f}  analytical={ana:.6f}  error={abs(num-ana):.2e}")

## 2. Multivariate Gradient

In [ ]:
def gradient(f, x, h=1e-5):
    """Compute gradient of f at point x (numpy array)."""
    grad = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        x_plus  = x.copy(); x_plus[i]  += h
        x_minus = x.copy(); x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad

# f(x,y) = x² + 2y²  →  ∇f = [2x, 4y]
f2d = lambda x: x[0]**2 + 2*x[1]**2
point = np.array([3.0, 2.0])

grad_num = gradient(f2d, point)
grad_ana = np.array([2*point[0], 4*point[1]])
print(f"Numerical gradient: {grad_num}")
print(f"Analytical gradient: {grad_ana}")

## 3. Chain Rule

In [ ]:
# Chain rule: d/dx f(g(x)) = f'(g(x)) * g'(x)
# Example: h(x) = sin(x²)
# h'(x) = cos(x²) * 2x

h = lambda x: np.sin(x**2)
dh_analytical = lambda x: np.cos(x**2) * 2 * x

x_vals = np.linspace(0, 3, 100)
dh_num = [numerical_gradient(h, x) for x in x_vals]
dh_ana = dh_analytical(x_vals)

plt.figure(figsize=(8, 4))
plt.plot(x_vals, h(x_vals), label="h(x) = sin(x²)")
plt.plot(x_vals, dh_ana, label="h'(x) = cos(x²)·2x  [analytical]")
plt.plot(x_vals, dh_num, "--", label="h'(x) [numerical]", alpha=0.7)
plt.legend()
plt.title("Chain Rule Verification")
plt.show()
print(f"Max error: {np.max(np.abs(np.array(dh_num) - dh_ana)):.2e}")

## 4. Gradient Descent — 1D

In [ ]:
def gradient_descent_1d(f, df, x0, lr=0.1, n_steps=50):
    x = x0
    history = [x]
    for _ in range(n_steps):
        x = x - lr * df(x)
        history.append(x)
    return x, history

# Minimize f(x) = (x-3)² + 1
f  = lambda x: (x - 3)**2 + 1
df = lambda x: 2 * (x - 3)

x_opt, history = gradient_descent_1d(f, df, x0=0.0, lr=0.2, n_steps=30)
print(f"Minimum found at x={x_opt:.6f} (true: x=3.0)")
print(f"f(x_opt) = {f(x_opt):.6f}")

x_plot = np.linspace(-1, 7, 200)
plt.figure(figsize=(8, 4))
plt.plot(x_plot, f(x_plot), label="f(x)")
plt.scatter(history, [f(x) for x in history], c=range(len(history)),
            cmap="plasma", zorder=3, label="GD steps")
plt.title("Gradient Descent on f(x) = (x-3)² + 1")
plt.legend()
plt.colorbar(label="step")
plt.show()

## 5. Learning Rate Effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
f  = lambda x: (x - 3)**2 + 1
df = lambda x: 2 * (x - 3)

for ax, (lr, label) in zip(axes, [(0.01, "Too small (lr=0.01)"),
                                    (0.3, "Good (lr=0.3)"),
                                    (1.1, "Too large (lr=1.1)")]):
    _, hist = gradient_descent_1d(f, df, x0=0.0, lr=lr, n_steps=30)
    losses = [f(x) for x in hist]
    ax.plot(losses, marker="o", markersize=3)
    ax.set_title(label)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_ylim(-0.5, 20)

plt.tight_layout()
plt.show()

## 6. Momentum & Adam Optimizer

In [ ]:
def gd_with_momentum(f, grad_f, x0, lr=0.1, beta=0.9, n_steps=100):
    x = np.array(x0, dtype=float)
    v = np.zeros_like(x)
    history = [x.copy()]
    for _ in range(n_steps):
        g = grad_f(x)
        v = beta * v + (1 - beta) * g
        x = x - lr * v
        history.append(x.copy())
    return x, history

def adam(f, grad_f, x0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, n_steps=100):
    x = np.array(x0, dtype=float)
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    history = [x.copy()]
    for t in range(1, n_steps+1):
        g = grad_f(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)   # bias correction
        v_hat = v / (1 - beta2**t)
        x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        history.append(x.copy())
    return x, history

# 2D function: Rosenbrock-like (non-convex)
f2 = lambda x: (x[0]-1)**2 + 10*(x[1]-x[0]**2)**2
grad_f2 = lambda x: np.array([
    2*(x[0]-1) - 40*x[0]*(x[1]-x[0]**2),
    20*(x[1]-x[0]**2)
])

x0 = [-1.0, 2.0]
opt_gd, hist_gd   = gradient_descent_1d.__wrapped__ if hasattr(gradient_descent_1d, '__wrapped__') else (None, None)  # skip

opt_mom, hist_mom = gd_with_momentum(f2, grad_f2, x0, lr=0.002, n_steps=500)
opt_adam, hist_adam = adam(f2, grad_f2, x0, lr=0.01, n_steps=500)

print(f"Momentum final: {opt_mom}  loss={f2(opt_mom):.4f}")
print(f"Adam final:     {opt_adam}  loss={f2(opt_adam):.6f}")
print(f"True minimum:   [1.0, 1.0]  loss=0")

## Interview Tips

- **Learning rate is the most important hyperparameter** — use LR schedulers in practice.
- **Adam** combines momentum (first moment) and RMSProp (second moment). Default lr=1e-3.
- **Gradient vanishing**: in deep nets, gradients shrink exponentially → use ReLU, batch norm, skip connections.
- **Gradient exploding**: gradients grow exponentially → gradient clipping (`clip_grad_norm_`).
- **Convex functions**: any local minimum is global. Most DL objectives are non-convex.
- **Batch GD vs SGD vs mini-batch**: mini-batch (32-256) balances noise and compute efficiency.